In [1]:
pip install -U beautifulsoup4 lxml requests

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


In [2]:
# ISKANDAR HAKIM BIN BURHANUDDIN (SW01083510) SECTION 01AT
# AHMAD ILYAS BIN AHMAD IRFAN (SW01083514) SECTION 01AT

import requests
import time
from bs4 import BeautifulSoup
import csv
# Why need Selenium?
# 1. website need user interact / click / scroll = JS rendered
# 2. some data not in initial HTML
# 3. handle dynamic loading (load more)
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Function to scrape reviews from Lazada review page (current DOM)
# Each review is in div.item; name=span.reviewer, date=span.time, text=item-content-main-content-reviews-item

def scrape_page(soup, reviews):
    for review_block in soup.find_all('div', class_='item'):
        # Reviewer name
        name_el = review_block.find('span', class_='reviewer')
        name = name_el.get_text(strip=True) if name_el else ''
        # Review date
        date_el = review_block.find('span', class_='time')
        date = date_el.get_text(strip=True) if date_el else ''
        # Review text
        text_container = review_block.find('div', class_='item-content-main-content-reviews-item')
        text = text_container.get_text(strip=True) if text_container else ''
        reviews.append({'Text': text, 'Author': name, 'Date': date})

# Base URL and headers
base_url = 'https://www.lazada.com.my/products/pdp-i2397262211-s10327297880.html'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36'}

# List to store reviews
reviews = []

# Scrape first 5 pages of Lazada reviews using Selenium pagination
def scrape_all_pages(url, max_pages=6):
    driver = webdriver.Chrome()
    try:
        driver.get(url)

        current_page = 1
        while current_page <= max_pages:
            # Wait until review items are present on the page
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.CLASS_NAME, 'item'))
            )

            # Parse current page
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            scrape_page(soup, reviews)

            # Stop if we've reached the target page count
            if current_page == max_pages:
                break

            # Find and click the "Next Page" button
            try:
                next_btn = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.CLASS_NAME, 'iweb-pagination-next'))
                )
                next_btn.click()
                current_page += 1
            except Exception:
                # No more pages or button not clickable
                break
    finally:
        driver.quit()

# 1. Selenium opens a real browser (Chrome).
# 2. The browser loads the page and runs the JavaScript.
# 3. The JavaScript fetches review data (often from an API) and inserts it into the page.
# 4. After that, the DOM contains the div.review-item elements.
# 5. Selenium gives you driver.page_source the fully rendered HTML.
# 6. BeautifulSoup parses that and finds the reviews.

In [3]:
# Scrape reviews from all pages
scrape_all_pages(base_url, max_pages=6)

WebDriverException: Message: unknown error: Chrome failed to start: exited abnormally.
  (unknown error: DevToolsActivePort file doesn't exist)
  (The process started from chrome location /usr/bin/chromium is no longer running, so ChromeDriver is assuming that Chrome has crashed.)
Stacktrace:
#0 0x60bbaa71fe89 <unknown>


In [ ]:
# Save quotes to CSV file
with open('review-lazada-extracted.csv', 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=['Text', 'Author', 'Date'])
    writer.writeheader()
    writer.writerows(reviews)